# SWYNEX Task 2 — Exploratory Data Analysis (EDA)

**Dataset:** `ecommerce_retail_transactions_cleaned.csv`  
**Tools:** Python, pandas, matplotlib  
**Goal:** Summarize the dataset, explore trends and patterns, visualize key measures, and document actionable insights.

## Metric definitions
- **Gross sales (estimated):** `quantity × unit_price_usd`
- **Discount amount (estimated):** `gross_sales_usd × discount_percent / 100`
- **Net sales (estimated):** `gross_sales_usd − discount_amount_usd`

These are analytical estimates from the fields available. They do not account for tax, shipping, returns, or other adjustments unless already included in the data. A row is treated as one transaction record; verify the source data's business definition before interpreting the measures as finalized accounting revenue.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Put the cleaned CSV in the same folder as this notebook, or update this path.
DATA_PATH = "ecommerce_retail_transactions_cleaned.csv"
df = pd.read_csv(DATA_PATH)
df.head()


In [ ]:
print("Rows, columns:", df.shape)
print("\\nColumn types:")
print(df.dtypes)
print("\\nMissing values:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count"))
print("\\nDuplicate rows:", df.duplicated().sum())
print("Unique order IDs:", df["order_id"].nunique())
print("Unique customers:", df["customer_id"].nunique())


In [ ]:
df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")
df["gross_sales_usd"] = df["quantity"] * df["unit_price_usd"]
df["discount_amount_usd"] = df["gross_sales_usd"] * df["discount_percent"] / 100
df["net_sales_usd"] = df["gross_sales_usd"] - df["discount_amount_usd"]
df["order_month"] = df["order_date"].dt.to_period("M").astype(str)

numeric_cols = ["quantity", "unit_price_usd", "discount_percent",
                "gross_sales_usd", "discount_amount_usd", "net_sales_usd",
                "customer_rating"]
display(df[numeric_cols].describe().T.round(2))


In [ ]:
# Category summary
category_summary = df.groupby("product_category").agg(
    transactions=("order_id", "count"),
    units=("quantity", "sum"),
    gross_sales_usd=("gross_sales_usd", "sum"),
    discounts_usd=("discount_amount_usd", "sum"),
    net_sales_usd=("net_sales_usd", "sum"),
    avg_rating=("customer_rating", "mean")
).sort_values("net_sales_usd", ascending=False)
display(category_summary.round(2))


In [ ]:
category_summary["net_sales_usd"].sort_values().plot(kind="barh", figsize=(10, 5))
plt.title("Net Sales by Product Category")
plt.xlabel("Estimated net sales (USD)")
plt.ylabel("Product category")
plt.tight_layout()
plt.show()


In [ ]:
# Monthly trend
monthly_summary = df.groupby("order_month").agg(
    transactions=("order_id", "count"),
    net_sales_usd=("net_sales_usd", "sum"),
    units=("quantity", "sum")
).reset_index()
monthly_summary.plot(x="order_month", y="net_sales_usd", kind="line", marker="o", figsize=(11, 5), legend=False)
plt.title("Monthly Estimated Net Sales")
plt.xlabel("Order month")
plt.ylabel("Estimated net sales (USD)")
plt.xticks(rotation=60, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
# Payment method and order status
payment_summary = df.groupby("payment_method").agg(
    transactions=("order_id", "count"),
    net_sales_usd=("net_sales_usd", "sum")
).sort_values("transactions", ascending=False)
display(payment_summary.round(2))
payment_summary["transactions"].sort_values().plot(kind="barh", figsize=(9, 5))
plt.title("Transactions by Payment Method")
plt.xlabel("Transactions")
plt.ylabel("Payment method")
plt.tight_layout()
plt.show()

status_summary = df.groupby("order_status").agg(
    transactions=("order_id", "count"),
    net_sales_usd=("net_sales_usd", "sum")
).sort_values("transactions", ascending=False)
display(status_summary.round(2))
status_summary["transactions"].sort_values().plot(kind="barh", figsize=(9, 5))
plt.title("Order Status Distribution")
plt.xlabel("Transactions")
plt.ylabel("Order status")
plt.tight_layout()
plt.show()


In [ ]:
# Ratings: analyze only records where a rating exists
ratings = df["customer_rating"].dropna()
print(f"Rated records: {len(ratings):,} / {len(df):,}")
print(f"Missing ratings: {df['customer_rating'].isna().sum():,} ({df['customer_rating'].isna().mean()*100:.1f}%)")
print(f"Mean rating among rated records: {ratings.mean():.2f}/5")
ratings.value_counts().sort_index().plot(kind="bar", figsize=(8, 4))
plt.title("Customer Rating Distribution (Rated Transactions Only)")
plt.xlabel("Rating")
plt.ylabel("Number of ratings")
plt.tight_layout()
plt.show()


In [ ]:
# Geographic and product summaries
country_summary = df.groupby("country").agg(
    transactions=("order_id", "count"),
    net_sales_usd=("net_sales_usd", "sum")
).sort_values("net_sales_usd", ascending=False)
display(country_summary.head(10).round(2))
country_summary.head(10).sort_values("net_sales_usd")["net_sales_usd"].plot(kind="barh", figsize=(9, 5))
plt.title("Top 10 Countries by Estimated Net Sales")
plt.xlabel("Estimated net sales (USD)")
plt.ylabel("Country")
plt.tight_layout()
plt.show()

product_summary = df.groupby("product_name").agg(
    transactions=("order_id", "count"),
    units=("quantity", "sum"),
    net_sales_usd=("net_sales_usd", "sum")
).sort_values("net_sales_usd", ascending=False)
display(product_summary.head(10).round(2))


## Insight checklist
Use the tables and plots above to report at least five evidence-backed insights. For each insight, state the metric, the actual value, why it matters, and a cautious business recommendation.

Remember:
- A high transaction count is not automatically high profitability.
- Ratings describe only the subset with recorded ratings.
- A status distribution is a snapshot unless the dataset's status history is known.
- Missing city data limits city-level comparisons.
- Net sales here are estimated from quantity, unit price, and discount only.
